## Calculating Energies

When we do long Caldeira Leggett simulations, it can become a lot of data to store if we want to keep track of the full wavefunction. To overcome this we can just keep track of specific parameters, such as the energy or location of the wavefunction.


In [ ]:
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
from scipy.constants import Boltzmann  # type: ignore stubs
from slate_core import Array, Basis, FundamentalBasis, TupleBasis
from slate_core.metadata import (
    Domain,
    spaced_volume_metadata_from_stacked_delta_x,
)
from slate_core.util import cached

from slate_quantum import operator, state
from slate_quantum.dynamics import (
    LangevinParameters,
    caldeira_leggett,
    langevin,
)
from slate_quantum.metadata import SpacedTimeMetadata

if TYPE_CHECKING:
    from slate_quantum.dynamics._realization import RealizationListIndexMetadata

metadata = spaced_volume_metadata_from_stacked_delta_x((np.array([2 * np.pi]),), (31,))

potential_single = operator.build.cos_potential(metadata, 1)
potential = operator.build.repeat_potential(potential_single, (3,))
potential_metadata = potential.basis.inner.outer_recast.metadata()


initial_state = state.build_coherent(potential_metadata, (0,), (0,), sigma_0=(0.5,))


parameters = LangevinParameters(mass=1, lambda_=0, temperature=8 / Boltzmann, hbar=1)
times = FundamentalBasis(SpacedTimeMetadata(600, domain=Domain(delta=60 * np.pi)))


@cached(Path("data/caldeira_leggett_positions.npz"))
def _simulate_quantum() -> tuple[
    Array[
        TupleBasis[tuple[FundamentalBasis, Basis[SpacedTimeMetadata]], None],
        np.dtype[np.complexfloating],
    ],
    Array[
        TupleBasis[tuple[FundamentalBasis, Basis[SpacedTimeMetadata]], None],
        np.dtype[np.floating],
    ],
]:
    return caldeira_leggett.solve_periodic_locations(
        initial_state,
        times,
        parameters,
        potential,
        method="Rouchon",
        target_delta=1e-3,
    )


@cached(Path("data/langevin_positions.npz"))
def _simulate_classical() -> Array[
    Basis[RealizationListIndexMetadata[SpacedTimeMetadata]],
    np.dtype[np.complexfloating],
]:
    return langevin.solve_periodic_langevin(
        0,
        times,
        parameters,
        potential_single,
        target_delta=0.5e-3,
    )


In [ ]:
from slate_core.plot import array_against_basis

alpha_quantum, norms = _simulate_quantum()

positions_quantum, momenta_quantum = parameters.eval_xp(alpha_quantum)

fig, ax, line = array_against_basis(norms[0, :])
ax.set_xlabel("Time / s")
ax.set_ylabel("Norm")
ax.set_title("Norm of the state")


fig, ax, line = array_against_basis(positions_quantum[0, :], measure="real")
ax.set_xlabel("Time / s")
ax.set_ylabel("Position")
ax.set_title("Position of the state")

fig, ax, line = array_against_basis(momenta_quantum[0, :], measure="real")
ax.set_xlabel("Time / s")
ax.set_ylabel("Momentum")
ax.set_title("Momentum of the state")

In [ ]:
from slate_core.plot import array_against_basis

alpha_classical = _simulate_classical()
positions_classical, momenta_classical = parameters.eval_xp(alpha_classical)


fig, ax, line = array_against_basis(positions_classical[0, :], measure="real")
ax.set_xlabel("Time / s")
ax.set_ylabel("Position")
ax.set_title("Position of the state")

fig, ax, line = array_against_basis(momenta_classical[0, :], measure="real")
ax.set_xlabel("Time / s")
ax.set_ylabel("Momentum")
ax.set_title("Momentum of the state")